# CNN_Baseline

In [1]:
# ==========================================
# COMPLETE CNN ANOMALY DETECTION PIPELINE
# FIXED + DEBUG READY VERSION
# ==========================================

import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

# load_and_concat/create_windows/SensorDataset used to be redefined in every
# notebook (03/04/05) with small drifts between copies - now a single shared
# implementation in src/core/windowing.py and src/models/cnn/model.py.
from src.core.windowing import load_split as load_and_concat, create_windows
from src.models.cnn.model import SensorDataset

# =========================
# Reproducibility
# =========================
torch.manual_seed(42)
np.random.seed(42)

# =========================
# Paths
# =========================
TRAIN_PATH = "../data/raw/train"
VAL_PATH = "../data/raw/val"
VAL_LABELS_PATH = "../data/raw/val_labels"
TEST_PATH = "../data/raw/test"

OUTPUT_CSV_PATH = "../outputs/cnn/predictions/test_predictions.csv"
# =========================
# Hyperparameters
# =========================
WINDOW_SIZE = 200
BATCH_SIZE = 64

# =========================
# CNN Autoencoder (simple baseline architecture used for this notebook's
# window-size experiment below - superseded by the BatchNorm version in the
# "Model Upgrade" section further down)
# =========================
class ConvAutoencoder1D(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_features, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(32, n_features, 3, stride=2, padding=1, output_padding=1)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# =========================
# Reconstruction Error
# =========================
def get_errors(model, loader, device):
    model.eval()
    errors = []

    loss_fn = nn.MSELoss(reduction='none')

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)

            loss = loss_fn(out, batch)
            loss = loss.mean(dim=[1,2]).cpu().numpy()

            errors.extend(loss)

    return np.array(errors)

# =========================
# MAIN PIPELINE
# =========================
print("Loading data...")

train_df = load_and_concat(TRAIN_PATH)
val_df = load_and_concat(VAL_PATH)
test_df = load_and_concat(TEST_PATH)

val_labels_df = load_and_concat(VAL_LABELS_PATH)

# ✅ FIXED LABEL COLUMN (IMPORTANT)
val_labels = val_labels_df["label"].values

print("Scaling data...")

scaler = StandardScaler()
train = scaler.fit_transform(train_df.values)
val = scaler.transform(val_df.values)
test = scaler.transform(test_df.values)

print("Creating windows...")

X_train, _ = create_windows(train, window_size=WINDOW_SIZE, step=10)
X_val, y_val = create_windows(val, val_labels, window_size=WINDOW_SIZE, step=1)
X_test, _ = create_windows(test, window_size=WINDOW_SIZE, step=1)

print("Shapes:")
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

train_loader = DataLoader(SensorDataset(X_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SensorDataset(X_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(SensorDataset(X_test), batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = ConvAutoencoder1D(X_train.shape[2]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# =========================
# TRAINING
# =========================
print("Training started...")

EPOCHS = 10  # keep small for debugging

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(train_loader):.6f}")

# =========================
# VALIDATION
# =========================
print("Validation...")

val_errors = get_errors(model, val_loader, device)

prec, rec, thr = precision_recall_curve(y_val, val_errors)
f1 = (2 * prec * rec) / (prec + rec + 1e-8)

best = np.argmax(f1)
best_threshold = thr[best]

print("Best Threshold:", best_threshold)

# =========================
# TEST PREDICTION (FIXED PART)
# =========================
print("Testing...")

test_errors = get_errors(model, test_loader, device)
test_predictions = (test_errors > best_threshold).astype(int)

# =========================
# SAVE OUTPUT
# =========================
output = pd.DataFrame({
    "Window_ID": np.arange(len(test_predictions)),
    "Reconstruction_Error": test_errors,
    "Is_Anomaly": test_predictions
})

output.to_csv(OUTPUT_CSV_PATH, index=False)

print("Saved to:", OUTPUT_CSV_PATH)
print("DONE ✔")


Loading data...
Scaling data...
Creating windows...
Shapes:
Train: (18925, 200, 18)
Val: (79489, 200, 18)
Test: (315825, 200, 18)
Device: cpu
Training started...
Epoch 1/10 Loss: 0.215390
Epoch 2/10 Loss: 0.045903
Epoch 3/10 Loss: 0.028910
Epoch 4/10 Loss: 0.020432
Epoch 5/10 Loss: 0.016443
Epoch 6/10 Loss: 0.013872
Epoch 7/10 Loss: 0.011768
Epoch 8/10 Loss: 0.010213
Epoch 9/10 Loss: 0.009010
Epoch 10/10 Loss: 0.008061
Validation...
Best Threshold: 0.0025890071
Testing...
Saved to: ../outputs/cnn/predictions/test_predictions.csv
DONE ✔


# Experiment 1: Window Size Optimization

In this step, we evaluate how different window sizes affect model performance.

Since this is a time-series anomaly detection problem, window size directly controls:

- How much temporal context the model can see
- Whether anomalies appear as local spikes or long-term drifts
- Model sensitivity vs stability trade-off

We will keep everything constant (model, epochs, scaler) and only vary:

> Window Size = [50, 100, 150, 200]

This ensures a fair comparison of temporal context impact on F1-score.

In [2]:
import numpy as np
from sklearn.metrics import f1_score

window_sizes = [50, 100, 150, 200]

results = []

for W in window_sizes:
    print("\n" + "="*50)
    print(f"RUNNING WINDOW SIZE: {W}")
    print("="*50)

    # =========================
    # 1. CREATE WINDOWS
    # =========================
    X_train, _ = create_windows(train, window_size=W, step=10)
    X_val, y_val = create_windows(val, val_labels, window_size=W, step=1)

    print("Train shape:", X_train.shape)
    print("Val shape:", X_val.shape)

    # =========================
    # 2. DATA LOADERS
    # =========================
    train_loader = DataLoader(SensorDataset(X_train), batch_size=64, shuffle=True)
    val_loader = DataLoader(SensorDataset(X_val), batch_size=64)

    # =========================
    # 3. NEW MODEL FOR EACH RUN
    # =========================
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = ConvAutoencoder1D(X_train.shape[2]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = torch.nn.MSELoss()

    # =========================
    # 4. TRAIN
    # =========================
    EPOCHS = 5  # keep small for experiments

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()
            out = model(batch)
            loss = loss_fn(out, batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(train_loader):.6f}")

    # =========================
    # 5. VALIDATION ERRORS
    # =========================
    val_errors = get_errors(model, val_loader, device)

    # =========================
    # 6. BEST THRESHOLD + F1
    # =========================
    thresholds = np.linspace(val_errors.min(), val_errors.max(), 100)

    best_f1 = 0
    best_th = 0

    for th in thresholds:
        preds = (val_errors > th).astype(int)
        score = f1_score(y_val, preds)

        if score > best_f1:
            best_f1 = score
            best_th = th

    print(f"Best F1 for window {W}: {best_f1:.4f}")
    print(f"Best threshold: {best_th:.6f}")

    results.append((W, best_f1, best_th))

# =========================
# FINAL SUMMARY
# =========================
print("\n================ FINAL RESULTS ================")

for r in results:
    print(f"Window: {r[0]} | F1: {r[1]:.4f} | Threshold: {r[2]:.6f}")


RUNNING WINDOW SIZE: 50
Train shape: (18940, 50, 18)
Val shape: (79639, 50, 18)
Epoch 1/5 Loss: 0.204504
Epoch 2/5 Loss: 0.048805
Epoch 3/5 Loss: 0.032718
Epoch 4/5 Loss: 0.024114
Epoch 5/5 Loss: 0.018850
Best F1 for window 50: 0.4086
Best threshold: 0.003908

RUNNING WINDOW SIZE: 100
Train shape: (18935, 100, 18)
Val shape: (79589, 100, 18)
Epoch 1/5 Loss: 0.214462
Epoch 2/5 Loss: 0.048470
Epoch 3/5 Loss: 0.031624
Epoch 4/5 Loss: 0.023101
Epoch 5/5 Loss: 0.018479
Best F1 for window 100: 0.4300
Best threshold: 0.003599

RUNNING WINDOW SIZE: 150
Train shape: (18930, 150, 18)
Val shape: (79539, 150, 18)
Epoch 1/5 Loss: 0.213634
Epoch 2/5 Loss: 0.044500
Epoch 3/5 Loss: 0.028048
Epoch 4/5 Loss: 0.020760
Epoch 5/5 Loss: 0.017038
Best F1 for window 150: 0.4494
Best threshold: 0.003384

RUNNING WINDOW SIZE: 200
Train shape: (18925, 200, 18)
Val shape: (79489, 200, 18)
Epoch 1/5 Loss: 0.209745
Epoch 2/5 Loss: 0.045629
Epoch 3/5 Loss: 0.026747
Epoch 4/5 Loss: 0.019599
Epoch 5/5 Loss: 0.016280


Tuning statrts from here , Above this is the Baseline for cnn!


# Step 3: Model Upgrade – Dilated Convolutional Autoencoder

From the window experiments, we observed that performance improves consistently as window size increases.

This confirms that the dataset contains long-term temporal dependencies (slow drift anomalies rather than sudden spikes).

To capture these long-range patterns more efficiently, we upgrade the model architecture:

## Why upgrade?
- Standard CNN uses pooling → loses temporal resolution
- Large windows increase computation cost
- We need long memory WITHOUT increasing window size further

## Solution: Dilated Convolutions

Dilated convolutions allow the model to:
- Expand receptive field exponentially
- Capture long-term dependencies
- Maintain computational efficiency

This step replaces the baseline CNN with a stronger temporal feature extractor while keeping the same anomaly detection framework (reconstruction error).

In [13]:
# ==========================================
# IMPROVED CNN AUTOENCODER (NO HYBRID)
# Stable + Better F1 version
# ==========================================

import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve

# =========================
# Reproducibility
# =========================
torch.manual_seed(42)
np.random.seed(42)

# =========================
# Paths
# =========================
TRAIN_PATH = "../data/raw/train"
VAL_PATH = "../data/raw/val"
VAL_LABELS_PATH = "../data/raw/val_labels"
TEST_PATH = "../data/raw/test"

OUTPUT_PATH = "../outputs/cnn/predictions.csv"

# =========================
# Hyperparameters
# =========================
WINDOW_SIZE = 200
BATCH_SIZE = 64
EPOCHS = 10

# =========================
# Load Data
# =========================
def load_folder(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if len(files) == 0:
        raise ValueError(f"No CSV files found in {folder}")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# =========================
# Windowing
# =========================
def create_windows(data, labels=None, window_size=200, step=1):
    X, y = [], []

    for i in range(0, len(data) - window_size + 1, step):
        X.append(data[i:i+window_size])

        if labels is not None:
            y.append(1 if np.mean(labels[i:i+window_size]) > 0 else 0)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32) if labels is not None else None

    return X, y

# =========================
# Dataset
# =========================
class SensorDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]

# =========================
# IMPROVED CNN AUTOENCODER
# =========================
class ConvAutoencoder1D(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_features, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.MaxPool1d(2)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, n_features, kernel_size=3, padding=1)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# =========================
# Error computation
# =========================
def get_errors(model, loader, device):
    model.eval()
    errors = []

    loss_fn = nn.MSELoss(reduction='none')

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)

            loss = loss_fn(out, batch)
            loss = loss.mean(dim=[1,2]).cpu().numpy()

            errors.extend(loss)

    return np.array(errors)

# =========================
# MAIN
# =========================
print("Loading data...")

train_df = load_folder(TRAIN_PATH)
val_df = load_folder(VAL_PATH)
test_df = load_folder(TEST_PATH)
val_labels = load_folder(VAL_LABELS_PATH)["label"].values

print("Scaling...")

scaler = StandardScaler()
train = scaler.fit_transform(train_df.values)
val = scaler.transform(val_df.values)
test = scaler.transform(test_df.values)

print("Windowing...")

X_train, _ = create_windows(train, window_size=WINDOW_SIZE, step=10)
X_val, y_val = create_windows(val, val_labels, window_size=WINDOW_SIZE, step=1)
X_test, _ = create_windows(test, window_size=WINDOW_SIZE, step=1)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

train_loader = DataLoader(SensorDataset(X_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SensorDataset(X_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(SensorDataset(X_test), batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = ConvAutoencoder1D(X_train.shape[2]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# =========================
# TRAINING
# =========================
print("Training...")

for epoch in range(EPOCHS):
    model.train()
    total = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch)

        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total/len(train_loader):.6f}")

# =========================
# VALIDATION
# =========================
print("Validation...")

val_errors = get_errors(model, val_loader, device)

prec, rec, thr = precision_recall_curve(y_val, val_errors)

# FIX: safe slicing (important)
f1 = 2 * prec * rec / (prec + rec + 1e-8)

best_idx = np.nanargmax(f1)

best_threshold = thr[max(best_idx-1, 0)]

print("Best F1:", f1[best_idx])
print("Best Threshold:", best_threshold)

# =========================
# TEST
# =========================
print("Testing...")

test_errors = get_errors(model, test_loader, device)
preds = (test_errors > best_threshold).astype(int)

# =========================
# CREATE CODABENCH SUBMISSION
# =========================

import glob
import os

test_files = sorted(glob.glob(os.path.join(TEST_PATH, "*.csv")))

submission_rows = []

pred_pointer = 0

for run_id, file_path in enumerate(test_files, start=1):

    df = pd.read_csv(file_path)
    n_timesteps = len(df)

    for timestep in range(n_timesteps):

        # map timestep -> nearest window prediction
        pred_idx = min(pred_pointer, len(preds) - 1)

        submission_rows.append({
            "run_id": run_id,
            "timestep": timestep,
            "prediction": int(preds[pred_idx])
        })

        pred_pointer += 1

submission = pd.DataFrame(
    submission_rows,
    columns=["run_id", "timestep", "prediction"]
)

submission.to_csv(OUTPUT_PATH, index=False)

print("Saved submission:")
print(submission.head())

print("\nShape:", submission.shape)
print("Unique run_ids:", submission["run_id"].nunique())

Loading data...
Scaling...
Windowing...
Shapes: (18925, 200, 18) (79489, 200, 18) (315825, 200, 18)
Device: cpu
Training...
Epoch 1/10 Loss: 0.139166
Epoch 2/10 Loss: 0.045815
Epoch 3/10 Loss: 0.036955
Epoch 4/10 Loss: 0.031917
Epoch 5/10 Loss: 0.029690
Epoch 6/10 Loss: 0.027140
Epoch 7/10 Loss: 0.024890
Epoch 8/10 Loss: 0.024775
Epoch 9/10 Loss: 0.022685
Epoch 10/10 Loss: 0.022348
Validation...
Best F1: 0.535037874191333
Best Threshold: 0.013939682
Testing...
Saved submission:
   run_id  timestep  prediction
0       1         0           1
1       1         1           1
2       1         2           1
3       1         3           1
4       1         4           1

Shape: (316024, 3)
Unique run_ids: 53


In [14]:
print(submission.groupby("run_id").size().head(10))

run_id
1     2584
2     4824
3     3390
4     6846
5     2774
6     5159
7     5140
8     4388
9     4611
10    4045
dtype: int64


In [12]:
import os
import glob
import pandas as pd

test_files = sorted(glob.glob(os.path.join(TEST_PATH, "*.csv")))

print("Number of test files:", len(test_files))

for i, f in enumerate(test_files[:10], start=1):
    df = pd.read_csv(f)
    print(i, os.path.basename(f), len(df))

Number of test files: 53
1 series_001.csv 2584
2 series_002.csv 4824
3 series_003.csv 3390
4 series_004.csv 6846
5 series_005.csv 2774
6 series_006.csv 5159
7 series_007.csv 5140
8 series_008.csv 4388
9 series_009.csv 4611
10 series_010.csv 4045


In [6]:
# ==========================================
# STRIDE TUNING EXPERIMENT (NO RELOAD VERSION)
# ==========================================

from sklearn.metrics import precision_recall_curve
import numpy as np

stride_values = [2, 5, 8, 10, 12]

results = []

for step in stride_values:
    print("\n" + "="*50)
    print(f"RUNNING STRIDE: {step}")
    print("="*50)

    # -------------------------
    # 1. Windowing (ONLY TRAIN CHANGES STRIDE)
    # -------------------------
    X_train_s, _ = create_windows(train, window_size=200, step=step)
    X_val_s, y_val_s = create_windows(val, val_labels, window_size=200, step=1)

    print("Train shape:", X_train_s.shape)
    print("Val shape:", X_val_s.shape)

    # -------------------------
    # 2. DataLoader
    # -------------------------
    train_loader_s = DataLoader(SensorDataset(X_train_s), batch_size=64, shuffle=True)
    val_loader_s = DataLoader(SensorDataset(X_val_s), batch_size=64, shuffle=False)

    # -------------------------
    # 3. NEW MODEL (reset each run)
    # -------------------------
    model = ConvAutoencoder1D(X_train_s.shape[2]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # -------------------------
    # 4. TRAIN (short debug epochs first)
    # -------------------------
    EPOCHS = 5

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for batch in train_loader_s:
            batch = batch.to(device)

            optimizer.zero_grad()
            out = model(batch)
            loss = loss_fn(out, batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(train_loader_s):.6f}")

    # -------------------------
    # 5. VALIDATION
    # -------------------------
    val_errors = get_errors(model, val_loader_s, device)

    prec, rec, thr = precision_recall_curve(y_val_s, val_errors)
    f1_scores = (2 * prec * rec) / (prec + rec + 1e-8)

    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_threshold = thr[best_idx]

    print(f"Best F1 for stride {step}: {best_f1:.4f}")
    print(f"Best threshold: {best_threshold:.6f}")

    results.append((step, best_f1, best_threshold))

# -------------------------
# FINAL SUMMARY
# -------------------------
print("\n\n========== STRIDE RESULTS ==========")
for r in results:
    print(f"Stride: {r[0]} | F1: {r[1]:.4f} | Threshold: {r[2]:.6f}")

best_stride = max(results, key=lambda x: x[1])
print("\n🏆 BEST STRIDE:", best_stride[0])
print("🏆 BEST F1:", best_stride[1])


RUNNING STRIDE: 2
Train shape: (94623, 200, 18)
Val shape: (79489, 200, 18)
Epoch 1/5 Loss: 0.056817
Epoch 2/5 Loss: 0.024756
Epoch 3/5 Loss: 0.020351
Epoch 4/5 Loss: 0.017332
Epoch 5/5 Loss: 0.015726
Best F1 for stride 2: 0.5264
Best threshold: 0.009658

RUNNING STRIDE: 5
Train shape: (37849, 200, 18)
Val shape: (79489, 200, 18)
Epoch 1/5 Loss: 0.095270
Epoch 2/5 Loss: 0.034639
Epoch 3/5 Loss: 0.028321
Epoch 4/5 Loss: 0.025597
Epoch 5/5 Loss: 0.023314
Best F1 for stride 5: 0.5112
Best threshold: 0.013631

RUNNING STRIDE: 8
Train shape: (23656, 200, 18)
Val shape: (79489, 200, 18)
Epoch 1/5 Loss: 0.124982
Epoch 2/5 Loss: 0.040187
Epoch 3/5 Loss: 0.032350
Epoch 4/5 Loss: 0.029455
Epoch 5/5 Loss: 0.028087
Best F1 for stride 8: 0.4796
Best threshold: 0.007769

RUNNING STRIDE: 10
Train shape: (18925, 200, 18)
Val shape: (79489, 200, 18)
Epoch 1/5 Loss: 0.136816
Epoch 2/5 Loss: 0.043582
Epoch 3/5 Loss: 0.036208
Epoch 4/5 Loss: 0.031539
Epoch 5/5 Loss: 0.028853
Best F1 for stride 10: 0.5180